# PEFT full run

Operational runbook for the long thread that produces the thesis's **PEFT** val
numbers: the (r, lr, epoch) sweep, the confirmation pass, the freeze, and the
frozen `peft` condition scored on full val. Runs on an **A100**.

Like the AFSP runbook it is organised into **four phases**, because the COMET
stack and the generation stack cannot share one Python environment
(`requirements-comet.txt` pins `transformers==4.57.6` / `numpy==1.26.4`, the
generation stack pins `transformers==5.12.1` / `numpy==2.4.1`), and the pipeline's
own ordering forces the alternation:

| Phase | Runtime | What runs | Depends on |
|------|---------|-----------|------------|
| 1 | **generation** (`requirements.txt`) | `peft_sweep` — train 4 cells x 3 epochs, eval_loss pre-filter, generate + score val candidates | — |
| 2 | **COMET** (`requirements-comet.txt`) | `peft_verify` (COMET + judge Φ), **freeze** the adapter | Phase 1 sweep result |
| 3 | **generation** (`requirements.txt`) | frozen `peft` condition on val + chrF/BLEU + stylometrics + judge Φ | frozen adapter |
| 4 | **COMET** (`requirements-comet.txt`) | `peft` COMET + paired bootstrap vs the ladder | Phase 3 outputs |

**Scale.** Full `train.jsonl` is 10,860 pairs; at effective batch 16 that is ~680
optimizer steps per epoch, ~2,040 per cell, x4 cells. With `--epochs-keep 2` the
sweep then generates 8 candidates x 1,323 val segments (~10.6k generations), and
Phase 2 regenerates the top 3 on full val. Order of a day of A100 time, so
**expect to span several sessions** — every stage is resumable (see below), and
Phase 1 must reach `results/peft_sweep_val.json` before Phase 2 can start.

**Resume rules** (all skip-if-exists, so re-running a cell is safe):
- training — a cell is skipped when its `models/.../epoch_checkpoints.json` exists (`--overwrite` retrains);
- candidate generation — skipped when `outputs/peft_sweep/<tag>_val.jsonl` exists;
- `--score-only` re-scores and re-ranks whatever is already on disk, training nothing;
- Phase 3 `infer` appends to `outputs/peft_val.jsonl` and verifies alignment on resume.

`data/splits/test.jsonl` stays sealed: the sweep refuses a `test` eval_file and
`peft_verify` refuses a `test` `--val-file`. Nothing here touches it.

---
## Phase 1 — generation runtime · the sweep

Fresh A100, default (generation) stack. This is the multi-session run that gates
everything downstream.

In [ ]:
# The sweep trains and then generates with Qwen2.5-7B in bf16 (~15 GB weights) —
# A100 40GB expected. Check the disk too: 4 cells x 3 kept epoch checkpoints
# (adapter + optimizer state) plus the base-model cache.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

In [ ]:
import os
if not os.path.isdir('Style-Aware-MT'):
    !git clone --branch feat/peft-implementation https://github.com/prnamhr/Style-Aware-MT.git
%cd Style-Aware-MT
!git rev-parse --short HEAD

In [ ]:
# Generation stack.
!pip install -r requirements.txt

# torchvision/torchaudio ship ABI-mismatched against the pinned torch and are
# unused by this text-only pipeline.
!pip uninstall -y torchvision torchaudio

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available(), torch.version.cuda)

### Persist the expensive artifacts across sessions

The checkpoints (`models/`) and the candidate generations
(`outputs/peft_sweep/`) are the run's cost; a lost session must not throw them
away. Symlink both to Drive so the skip-if-exists resume works on a fresh VM.
Set `PERSIST = False` to run without Drive (single-session only).

`results/` is deliberately *not* symlinked — it holds the committed
`stylometrics_centroid.json` that `register_fit` reads. The result JSONs are
copied to Drive at the end of each phase instead, and are anyway cheap to
rebuild with `--score-only`.

In [ ]:
PERSIST = True
DRIVE_ROOT = '/content/drive/MyDrive/style-aware-mt/peft'

import os, pathlib, shutil
if PERSIST:
    from google.colab import drive
    drive.mount('/content/drive')
    for rel in ('models', 'outputs/peft_sweep', 'results_backup'):
        target = pathlib.Path(DRIVE_ROOT) / rel
        target.mkdir(parents=True, exist_ok=True)
        if rel == 'results_backup':
            continue                                  # backup dir only, not linked in
        link = pathlib.Path(rel)
        if link.is_symlink():
            print(f'{link} -> {link.resolve()} (already linked)')
            continue
        if link.exists():
            # outputs/peft_sweep is a real, committed dir. Move what is in it onto
            # Drive and replace it with the link, or nothing written there persists.
            moved = 0
            for item in link.iterdir():
                dest = target / item.name
                if not dest.exists():
                    shutil.move(str(item), str(dest))
                    moved += 1
            shutil.rmtree(link)
            print(f'{link}: moved {moved} existing file(s) onto Drive, replacing dir with link')
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(target, target_is_directory=True)
        print(f'{link} -> {target}')
    !ls -la models outputs/peft_sweep | head -20
else:
    print('PERSIST=False — artifacts live on the VM only; a session loss restarts Phase 1.')

### Preflight: clear smoke artifacts out of the candidate directory

`outputs/peft_sweep/peft_r16_lr2e-4_e1_val.jsonl` is **committed** from the
32-row smoke run (`bc694f0`), and it sits at exactly the path this run's
anchor-epoch-1 candidate writes to. `generate_cell` skips a candidate whose file
already exists, so left in place it would be silently scored as a full-val
candidate. Delete any short file before training starts.

In [ ]:
import json, pathlib

VAL_N = sum(1 for line in open('data/splits/val.jsonl', encoding='utf-8') if line.strip())
print(f'full val = {VAL_N} segments\n')

for p in sorted(pathlib.Path('outputs/peft_sweep').glob('*_val.jsonl')):
    n = sum(1 for line in p.open(encoding='utf-8') if line.strip())
    if n < VAL_N:
        p.unlink()
        print(f'removed {p} ({n} rows — smoke/partial, would be reused as a candidate)')
    else:
        print(f'kept    {p} ({n} rows — full val)')

# Stale models/ from a smoke run would also be skipped as "already trained".
for man in sorted(pathlib.Path('models').glob('*/epoch_checkpoints.json')):
    print('\nexisting manifest:', man)
    print(man.read_text(encoding='utf-8'))

### The grid plan

`--dry-run` prints the 4 cells x 3 epochs it is about to train and where each
lands. Read it before committing the GPU hours.

In [ ]:
!python manage.py peft_sweep --config configs/peft_sweep.yaml --dry-run

### Train, pre-filter, generate, score

One command does the whole of Phase 1: train each cell for the full 3-epoch
budget keeping every epoch checkpoint, prune to the 2 lowest-`eval_loss` epochs
per cell (a *generation* pre-filter — it never picks the reported adapter),
generate those candidates on val, then rank them on the matched proxy axis
(chrF adequacy band + `register_fit` Φ proxy).

This is the long one. If the session dies, re-run this same cell — trained cells
and finished candidate files are skipped.

In [ ]:
!python manage.py peft_sweep --config configs/peft_sweep.yaml --epochs-keep 2

### Manifest sanity check

Each cell must have produced 3 distinct epoch checkpoints (the e3-duplicate bug
the multi-epoch smoke was written for), with a moving `eval_loss` — identical
losses across epochs means the model is not updating.

In [ ]:
import json
from pathlib import Path

manifests = sorted(Path('models').glob('peft_lora_*/epoch_checkpoints.json'))
print(f'{len(manifests)} cell manifest(s) found\n')
problems = []
for man_path in manifests:
    man = json.loads(man_path.read_text(encoding='utf-8'))
    epochs = [m['epoch'] for m in man]
    steps = [m['step'] for m in man]
    losses = [m['eval_loss'] for m in man]
    missing = [m['checkpoint'] for m in man if not m['checkpoint'] or not Path(m['checkpoint']).exists()]
    print(f"{man_path.parent.name:<26} epochs={epochs} steps={steps}")
    print('    eval_loss: ' + '  '.join(f'e{e}={l:.4f}' for e, l in zip(epochs, losses)))
    if epochs != [1, 2, 3]:
        problems.append(f'{man_path.parent.name}: epochs not 1..3 -> {epochs}')
    if len(set(steps)) != len(steps):
        problems.append(f'{man_path.parent.name}: duplicate checkpoint step -> {steps}')
    if len(set(round(l, 6) for l in losses)) <= 1:
        problems.append(f'{man_path.parent.name}: eval_loss flat across epochs -> {losses}')
    if missing:
        problems.append(f'{man_path.parent.name}: checkpoint dir(s) missing on disk -> {missing}')

assert len(manifests) == 4, f'expected 4 cells trained, found {len(manifests)}'
assert not problems, 'manifest problems:\n  ' + '\n  '.join(problems)
print('\nMANIFESTS OK: 4 cells x 3 distinct epoch checkpoints, eval_loss moving.')

### The proxy pick

Free/local proxies only (chrF + `register_fit`). It decides which candidates are
worth paying COMET and the judge for — not what gets reported.

In [ ]:
import json

sweep = json.load(open('results/peft_sweep_val.json'))
print(f"epochs_keep={sweep['epochs_keep']}  adequacy_margin={sweep['adequacy_margin']}  "
      f"candidates={len(sweep['cells'])}\n")
hdr = f"{'tag':<24}{'r':>4}{'a':>5}{'lr':>9}{'ep':>4}{'n':>6}{'chrF':>8}{'reg_fit':>9}{'eval_loss':>11}"
print(hdr); print('-' * len(hdr))
for c in sorted(sweep['cells'], key=lambda c: c.get('register_fit', 1e9)):
    print(f"{c['tag']:<24}{c['r']:>4}{c['alpha']:>5}{c['lr']:>9g}{c['epoch']:>4}{c['n']:>6}"
          f"{c['chrF']:>8}{c.get('register_fit', float('nan')):>9}{c['eval_loss']:>11.4f}")

rec = sweep.get('recommended')
print('\nproxy recommended:', rec and {k: rec[k] for k in ('tag', 'r', 'lr', 'epoch', 'chrF', 'register_fit') if k in rec})

# Every candidate must have been generated on FULL val, not a leftover short file.
short = [c['tag'] for c in sweep['cells'] if c['n'] != VAL_N]
assert not short, f'candidates not scored on full val ({VAL_N}): {short}'

In [ ]:
# Back the phase-1 result up to Drive before switching runtimes.
if PERSIST:
    !cp -v results/peft_sweep_val.json {DRIVE_ROOT}/results_backup/

---
## Phase 2 — COMET runtime · verify + freeze

The confirmation pass: regenerate the top 3 proxy candidates on full val (skipped
if already there), score COMET, run the judge for Φ, then freeze on the same rule
AFSP freezes on — COMET adequacy band, register fidelity decides inside it.

Installing the COMET pins downgrades `transformers`/`numpy`; **restart the
runtime is not needed, but do not run Phase 1 or 3 cells again until Phase 3
reinstalls `requirements.txt`.**

In [ ]:
!pip install -q -r requirements-comet.txt

In [ ]:
import os, getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')

In [ ]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
# COMET + judge Phi on the top 3. The per-segment judge cache under
# results/judge_val_segments/ is resumable, so a rerun does not re-pay.
!USE_TF=0 python -m src.peft.verify --config configs/peft_sweep.yaml \
    --judge-config configs/judge_eval.yaml --top 3

In [ ]:
# The freeze decision from the reported metrics.
import json

v = json.load(open('results/peft_verify_val.json'))
print('freeze tag :', v['freeze'],
      '(proxy pick held)' if v['proxy_pick_held'] else '(runner-up overtook the proxy pick)')
print('checkpoint :', v['freeze_checkpoint'], '\n')
for c in v['cells']:
    mark = '  <== freeze' if c['tag'] == v['freeze'] else ''
    phi = f"{c['judge_mean']:.3f}" if c['judge_mean'] is not None else 'n/a'
    print(f"  {c['tag']:<24} r={c['r']} lr={c['lr']:g} ep={c['epoch']}  "
          f"chrF {c['chrF']}  COMET {c['comet_system']:.4f}  Phi {phi}"
          f"  (judge cov {c['judge_coverage']}){mark}")

### Freeze the adapter into `configs/peft_qwen.yaml`

`generator.adapter_path` is what the `peft` inference condition loads. Point it
at the frozen checkpoint before generating the reported condition.

In [ ]:
import json, re, pathlib

v = json.load(open('results/peft_verify_val.json'))
ckpt = v['freeze_checkpoint']
assert ckpt and pathlib.Path(ckpt).exists(), f'frozen checkpoint missing on disk: {ckpt}'
frozen = next(c for c in v['cells'] if c['tag'] == v['freeze'])

p = pathlib.Path('configs/peft_qwen.yaml')
text = p.read_text(encoding='utf-8')
text, n = re.subn(r'(?m)^(\s*adapter_path:\s*)\S+', lambda m: f'{m.group(1)}{ckpt}', text, count=1)
assert n == 1, 'no generator.adapter_path line found in configs/peft_qwen.yaml'
p.write_text(text, encoding='utf-8')

print(f"froze generator.adapter_path = {ckpt}")
print(f"  (r={frozen['r']}, alpha={frozen['alpha']}, lr={frozen['lr']:g}, epoch={frozen['epoch']})")
!grep -n 'adapter_path:' configs/peft_qwen.yaml

In [ ]:
if PERSIST:
    !cp -v results/peft_verify_val.json {DRIVE_ROOT}/results_backup/

---
## Phase 3 — generation runtime · the frozen `peft` condition

Back to the generation stack. The `peft` condition uses the byte-identical
zero-shot prompt the adapter was trained on — no exemplars; the register is
carried by the adapter.

In [ ]:
!pip install -q -r requirements.txt
# Same torchvision/torchaudio ABI mismatch as Phase 1 — drop them (text-only pipeline).
!pip uninstall -y torchvision torchaudio

In [ ]:
# Full val with the frozen adapter; resumable via outputs/peft_val.jsonl.
!python manage.py infer --condition peft --config configs/peft_qwen.yaml

In [ ]:
# Adequacy proxies (chrF/BLEU) and register stylometrics — free/local, no COMET.
!python manage.py eval         --conditions peft --split val
!python manage.py stylometrics --conditions peft --split val

In [ ]:
# Register fidelity (judge Phi) for the reported PEFT row.
import os, getpass
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')
!python manage.py judge --conditions peft --split val --config configs/judge_eval.yaml

---
## Phase 4 — COMET runtime · PEFT COMET + paired bootstrap

The paired bootstrap needs the comparison conditions' `outputs/*_val.jsonl` from
the AFSP ladder run present in this session (Drive-restore them, or rerun the
AFSP runbook's Phase 3 first). The cell below reports which rungs it found and
bootstraps only against those — it does not silently drop a condition.

In [ ]:
!pip install -q -r requirements-comet.txt

In [ ]:
!python manage.py comet --conditions peft --split val

In [ ]:
import pathlib

LADDER = ['zeroshot', 'random_fewshot', 'knn_fewshot', 'afsp_margin', 'afsp_full']
present = [c for c in LADDER if pathlib.Path(f'outputs/{c}_val.jsonl').exists()]
absent = [c for c in LADDER if c not in present]
print('ladder outputs present:', present)
if absent:
    print('MISSING (excluded from the bootstrap):', absent)

In [ ]:
# PEFT vs the ladder rungs that are present, baseline = knn_fewshot when available.
conds = ' '.join(['peft'] + present)
baseline = 'knn_fewshot' if 'knn_fewshot' in present else 'peft'
!python manage.py bootstrap --metric comet --conditions {conds} --split val --baseline {baseline}

In [ ]:
if PERSIST:
    !cp -v results/*val*.json {DRIVE_ROOT}/results_backup/ 2>/dev/null; ls {DRIVE_ROOT}/results_backup

---
### What this run produces

| Artifact | Written by |
|---|---|
| `models/peft_lora_r{r}_lr{lr}/` + `epoch_checkpoints.json` | Phase 1 training (12 checkpoints) |
| `outputs/peft_sweep/<tag>_val.jsonl` | Phase 1 candidate generation |
| `results/peft_sweep_val.json` | Phase 1 proxy ranking |
| `results/peft_verify_val.json` | Phase 2 COMET + Φ, freeze decision |
| `configs/peft_qwen.yaml` (`generator.adapter_path`) | Phase 2 freeze |
| `outputs/peft_val.jsonl` | Phase 3 frozen condition on val |
| `results/eval_val.json`, `results/stylometrics_val.json`, `results/judge_val.json` | Phase 3 scoring |
| `results/comet_val.json`, `results/bootstrap_comet_val.json` | Phase 4 |

Commit the result JSONs and the frozen `configs/peft_qwen.yaml`, and log the run
in `docs/DEVLOG.md` (the 2026-07-23 PEFT entry records the implementation as
NOT run — this is the entry that closes it). `data/splits/test.jsonl` remains
sealed until every condition is frozen.